# 14 · M1.9 — M1.8 with a trend scenario: 1 draw in 4 carries the window's full trend

## Executive summary (read this first)

**M1.9 = M1.8 + a second scenario in the same forecast.** M1.7's shrunk drift is the best single guess, but a steady trend often
keeps going, and a single guess cannot say "probably not, but maybe". So M1.9 mixes two scenarios:

- **3 draws in 4 (75 %):** M1.8 as it is, with the shrunk recent trend.
- **1 draw in 4 (25 %):** the full trend of the 260-step window, `last + h × mean step`, with no shrinkage.

CPI and NFP keep their 12-month drift in both groups. The shocks, the width, the floor and the correlation are the same in both.
This is **forecast combination** (a linear pool of two predictive distributions), the standard way to hedge between models.
It is not an average of the two centres: the mixture is wider toward the trend, and only there.

**How we got here** (§1 measures each of these): pooling a quarter of **M0's** draws into M1.8 helped (0.9266 → 0.9218).
The gain came from M0's drift: with it removed, the pool made things worse (0.9288). Our own model with the full trend does
better than M0's draws (0.9194), and stays self-contained.

**Results** (§3): M1.9 **0.9194** × M0 (M1.8 0.9266); better than M0 on 92 of 103 units (M1.8: 84). The share hardly matters between
15 % and 35 % (0.9206, 0.9194, 0.9208); one in four is kept.

§2 shows the two scenarios on one date. No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 13's M1.8, plus the trend scenario

`m19_draws(U, origin)` is M1.9; `trend_mix=0` is M1.8 (checked against notebook 13's saved scores). The other variants are the
evidence above: a quarter of M0's draws (with and without M0's drift) instead of our trend scenario, and the share at 15 % and
35 %. `m0_at` is M0 rebuilt at a date, as in notebook 01, on its own random numbers.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

# Every asset's whole history as steps (its own transform, M0's gap rule), computed once: the recent trend (M1.7) looks
# back 126 steps, the variance ratio (M1.8) over the whole history.
FULL = {}
for _u, _U in UNITS_DATA.items():
    for _a in _U["assets"]:
        _rows = _U["rows"][_a]
        _lm = _U["rules"][_a]["transform"] == "log_diff" and min(v for _, v in _rows) > 0
        _st, _ = m0._steps([(d, np.log(v)) for d, v in _rows] if _lm else _rows, "level" if _lm else _U["target_type"])
        _ds = sorted(_st)
        FULL[(_u, _a)] = (np.array([str(d) for d in _ds]), np.array([_st[d] for d in _ds]))

def hist_steps(U, a, origin):
    """All of one asset's steps up to the origin, oldest first."""
    d, v = FULL[(U["unit"], a)]
    return v[: np.searchsorted(d, origin, side="right")]

def recent_trend(U, a, origin, mu_window):
    """M1.7's drift per step.
    CPI, NFP (drift by type): the mean of the last 12 monthly steps -- this year's trend, not the 21-year average.
    Everything else: the mean m of the last 126 steps, shrunk toward 0 by its t-statistic t = m / (sd / sqrt(126))
    (James-Stein): drift = m x max(0, 1 - 4 / t^2). |t| <= 2 gives no drift at all; |t| = 4 keeps 75 %."""
    v = hist_steps(U, a, origin)
    if U["rules"][a]["drift"]:
        return float(v[-12:].mean()) if len(v) >= 12 else mu_window
    v = v[-126:]
    sd = v.std(ddof=1)
    t = v.mean() / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return float(v.mean() * max(0.0, 1.0 - 4.0 / max(t * t, 1e-12)))

def variance_ratio(U, a, h_steps, origin, n0=30.0):
    """M1.8: VR = mean(h-step move^2) / (h x mean(1-step move^2)) over the asset's whole history before the origin.
    VR > 1: moves add up faster than sqrt(h) (trends, cycles); VR < 1: they partly cancel (mean reversion). Raw moves (no
    demeaning) for assets without a drift, because a trend we do not forecast is width. Shrunk toward 1 by the number of
    independent windows: VR = 1 + (VR_raw - 1) x n / (n + 30), n = history steps / h."""
    v = hist_steps(U, a, origin)
    if h_steps <= 1 or len(v) < 2 * h_steps + 20:
        return 1.0
    m = v.mean() if U["rules"][a]["drift"] else 0.0
    c = np.concatenate([[0.0], np.cumsum(v - m)])
    ratio = np.mean((c[h_steps:] - c[:-h_steps]) ** 2) / (h_steps * np.mean((v - m) ** 2))
    n_eff = len(v) / h_steps
    return 1.0 + (ratio - 1.0) * n_eff / (n_eff + n0)

def m19_draws(U, origin, drift17=True, floor=True, vr=True, trend_mix=0.25, ewma_halflife=63):
    """M1.9 = M1.8 + a trend scenario: the last trend_mix of the draws carry the window's full trend.
    trend_mix=0 is M1.8 (notebook 13)."""
    N = m0.N_DRAWS
    by_asset, x, log_mode, last = fit(U, origin, WINDOW)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
    sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])     # M1.5: EWMA sd, halflife 63
    sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    drift = {a: (mu[ai[a]] if U["rules"][a]["drift"] else 0.0) for a in by_asset}      # M1.5: drift by type
    if drift17:                                                                          # M1.7: the recent trend
        drift = {a: recent_trend(U, a, origin, mu[ai[a]]) for a in by_asset}
    def centre(a, h, d):
        return (0.0 if log_mode[a] else last[a]) + s[h] * d[a]
    mean = np.array([centre(a, h, drift) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((N, len(cells))) @ f.T
    scale = np.ones(len(cells))
    if vr:
        scale = scale * np.array([np.sqrt(variance_ratio(U, a, s[h], origin)) for a, h in cells])
    dev = dev * scale
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(N, 1))), 1.0)
    sm = mean + dev
    if trend_mix:
        k = int(round(trend_mix * N))           # the last k draws: the full trend of the window
        full = {a: (drift[a] if U["rules"][a]["drift"] else mu[ai[a]]) for a in by_asset}
        sm[N - k:] = np.array([centre(a, h, full) for a, h in cells]) + dev[N - k:]
    out = np.empty((N, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if floor and asset_type(a) == "rates":
            col = np.maximum(col, 0.0)          # zero lower bound on Treasury yields
        if s[h] <= 0:
            col = np.full(N, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

def m0_at(U, origin, with_drift=True, n_draws=500):
    """M0 at a backtest date (docs/M0-BASELINE.md): last 300 rows, M0's step, drift, normal, its own random numbers."""
    tt = U["target_type"]; by = sorted(U["assets"]); steps, anchor = {}, {}
    for a in by:
        k = bisect.bisect_right(U["dates"][a], origin)
        st, lv = m0._steps(U["rows"][a][max(0, k - 300):k], tt); steps[a] = st; anchor[a] = lv if tt == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by)))
    x = np.array([[steps[a][d] for d in common] for a in by]); mu = x.mean(1) if with_drift else np.zeros(len(by))
    sigma = np.atleast_2d(np.cov(x)); ai = {a: i for i, a in enumerate(by)}
    cells = [(a, h) for a in by for h in sorted(U["horizons"])]; s = U["steps"]
    mean = np.array([anchor[a] + s[h] * mu[ai[a]] for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    sm = mean + np.random.default_rng([zlib.crc32(U["id"].encode()) & 0x7FFFFFFF, 3]).standard_normal((n_draws, len(cells))) @ f.T
    out = np.empty((n_draws, len(U["assets"]), len(U["horizons"])))
    for ci, (a, h) in enumerate(cells):
        out[:, U["assets"].index(a), U["horizons"].index(h)] = sm[:, ci]
    return out

def draw_variant(U, o, kw):
    kw = dict(kw)
    pool = kw.pop("pool_m0", None)
    d = m19_draws(U, o, **kw)
    if pool is not None:                     # the last quarter of the draws come from M0 instead
        k = int(round(0.25 * len(d))); d = np.concatenate([d[: len(d) - k], m0_at(U, o, with_drift=pool)[:k]])
    return d
VARIANTS = {"M1.8": dict(trend_mix=0.0), "M1.8 + 25 % of M0's draws": dict(trend_mix=0.0, pool_m0=True),
            "M1.8 + 25 % of M0's draws, M0 without its drift": dict(trend_mix=0.0, pool_m0=False),
            "M1.9 with 15 %": dict(trend_mix=0.15), "M1.9": {}, "M1.9 with 35 %": dict(trend_mix=0.35)}
def score_unit(u):
    U = UNITS_DATA[u]
    out = []
    for o, y in BACKTEST[u]:
        for m, kw in VARIANTS.items():
            d = draw_variant(U, o, kw)
            out.append({"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], d.reshape(d.shape[0], -1), y)})
    return out
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // len(VARIANTS):,} dates x {len(VARIANTS)} variants in {time.time() - t0:.0f}s")
SCV = {m: vs_m0(G[G.model == m]) for m in VARIANTS}
d = float((SCV["M1.8"] - vs_m0(pd.read_parquet(DATA / "m1_8_backtest_scores.parquet"))).abs().max())
print(f"check — the new pieces switched off vs notebook 13's saved M1.8: max |difference| per unit = {d:.1e}")
assert d < 1e-12
print(pd.Series({m: s.mean() for m, s in SCV.items()}, name="competition score vs M0 (all units)").round(4).to_string())
M19 = G[G.model == "M1.9"].drop(columns="model").reset_index(drop=True)

scored 23,635 dates x 6 variants in 18s
check — the new pieces switched off vs notebook 13's saved M1.8: max |difference| per unit = 0.0e+00
M1.8                                              0.9266
M1.8 + 25 % of M0's draws                         0.9218
M1.8 + 25 % of M0's draws, M0 without its drift   0.9288
M1.9 with 15 %                                    0.9206
M1.9                                              0.9194
M1.9 with 35 %                                    0.9208


## 2 — Worked example: the two scenarios on one date

The 2-year yield in the 2016–18 hiking cycle (t2-F2-powell-pivot-2019, horizon 126), on its strongest-trend backtest date.
375 draws are centred on `last + 126 × M1.7's drift`, 125 on `last + 126 × the window's mean step`.

In [3]:
def last_value(U, a, o):
    return U["rows"][a][bisect.bisect_right(U["dates"][a], o) - 1][1]
u = "t2-F2-powell-pivot-2019"; U = UNITS_DATA[u]; a = "UST_2Y"
def t_of(o):
    v = hist_steps(U, a, o)[-126:]; return abs(v.mean() / (v.std(ddof=1) / np.sqrt(len(v))))
o, y = max(BACKTEST[u], key=lambda oy: t_of(oy[0]))
_, x, _, _ = fit(U, o, WINDOW); last = last_value(U, a, o)
shrunk, full = recent_trend(U, a, o, x[0].mean()), x[0].mean()
display(pd.Series({"date": o, "last value": last, "M1.7 drift per step (last 126 steps, shrunk)": shrunk,
                   "trend scenario per step (mean of the 260-step window)": full,
                   "centre of 375 draws = last + 126 × shrunk": last + 126 * shrunk,
                   "centre of 125 draws = last + 126 × full": last + 126 * full, "answer at 126": y[0]}, name=u).to_frame())
q = [0.05, 0.25, 0.5, 0.75, 0.95]
print("quantiles of the forecast at 126 steps:")
pd.DataFrame({"M1.8": np.quantile(m19_draws(U, o, trend_mix=0.0)[:, 0, 0], q), "M1.9": np.quantile(m19_draws(U, o)[:, 0, 0], q)},
             index=[f"{int(p * 100)} %" for p in q]).assign(answer=y[0])

,t2-F2-powell-pivot-2019
date,2018-03-14
last value,2.26
"M1.7 drift per step (last 126 steps, shrunk)",0.005023
trend scenario per step (mean of the 260-step window),0.003731
centre of 375 draws = last + 126 × shrunk,2.893
centre of 125 draws = last + 126 × full,2.73
answer at 126,2.74


quantiles of the forecast at 126 steps:


,M1.8,M1.9,answer
5 %,2.411,2.341,2.74
25 %,2.692,2.663,2.74
50 %,2.896,2.861,2.74
75 %,3.129,3.079,2.74
95 %,3.413,3.392,2.74


## 3 — Every model on the same backtest

In [4]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "production walk (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.5 (EWMA sd 63)": pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"),
          "M1.7 (recent trend)": pd.read_parquet(DATA / "m1_7_backtest_scores.parquet"),
          "M1.8 (+ floor, variance ratio)": pd.read_parquet(DATA / "m1_8_backtest_scores.parquet"),
          "M1.9 (+ trend scenario)": M19}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
production walk (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
M1.5 (EWMA sd 63),0.8932,0.99,0.9141,0.9642,0.9444,85/103
M1.7 (recent trend),0.8828,0.9656,0.9124,0.9505,0.9312,86/103
"M1.8 (+ floor, variance ratio)",0.8714,0.961,0.9097,0.9498,0.9266,84/103
M1.9 (+ trend scenario),0.8673,0.9461,0.9084,0.9428,0.9194,92/103


## 4 — M1.8 vs M1.9: per family, per asset type, and the units that moved most

In [5]:
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE), "M1.5 vs M0": SC["M1.5 (EWMA sd 63)"],
                         "M1.8 vs M0": SC["M1.8 (+ floor, variance ratio)"], "M1.9 vs M0": SC["M1.9 (+ trend scenario)"]}).loc[:, lambda d: ~d.columns.duplicated()]
def summary(g):
    return {"units": len(g), "M1.5": g["M1.5 vs M0"].mean(), "M1.8": g["M1.8 vs M0"].mean(), "M1.9": g["M1.9 vs M0"].mean(),
            "M1.9 better than M1.8": f'{(g["M1.9 vs M0"] < g["M1.8 vs M0"]).sum()}/{len(g)}',
            "M1.9 better than M0": f'{(g["M1.9 vs M0"] < 1).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)
moved = (PER_UNIT["M1.9 vs M0"] - PER_UNIT["M1.8 vs M0"]).sort_values()
print("the units that moved most (M1.9 - M1.8; below 0 = better):")
display(PER_UNIT.loc[list(moved.index[:8]) + list(moved.index[-5:])].assign(change=moved))

per family:


,units,M1.5,M1.8,M1.9,M1.9 better than M1.8,M1.9 better than M0
family,,,,,,
T2-F1,23,0.8932,0.8714,0.8673,11/23,20/23
T2-F2,27,0.99,0.961,0.9461,16/27,22/27
T2-F3,22,0.9141,0.9097,0.9084,12/22,22/22
T2-F4,31,0.9642,0.9498,0.9428,19/31,28/31
all,103,0.9444,0.9266,0.9194,58/103,92/103


per asset type:


,units,M1.5,M1.8,M1.9,M1.9 better than M1.8,M1.9 better than M0
asset type,,,,,,
cpi,2,0.899,0.735,0.735,0/2,2/2
equity_factor,16,0.9665,0.9673,0.9555,12/16,14/16
fx,35,0.9614,0.9447,0.94,17/35,31/35
fx+rates,5,0.933,0.9297,0.9174,5/5,5/5
payrolls,1,0.9682,0.8225,0.8225,0/1,1/1
rates,43,0.9234,0.9059,0.8983,23/43,39/43
unemployment,1,1.024,1.009,1.008,1/1,0/1


the units that moved most (M1.9 - M1.8; below 0 = better):


,family,asset type,M1.5 vs M0,M1.8 vs M0,M1.9 vs M0,change
t2-F2-cnh-tradewar-2018,T2-F2,fx,1.979,1.468,1.277,-0.191
t2-F2-powell-pivot-2019,T2-F2,rates,1.429,1.307,1.181,-0.1254
t2-F1-fed-put-2019,T2-F1,equity_factor,1.123,1.106,1.003,-0.1032
t2-F1-pause-2006,T2-F1,rates,1.291,1.233,1.132,-0.1012
t2-F4-hike-cycle-2021Q4b,T2-F4,rates,0.8661,0.7769,0.7105,-0.06639
t2-F2-trichet-alertness-2008,T2-F2,fx,1.128,1.12,1.06,-0.05987
t2-F1-conflicting-texts-2024,T2-F1,rates,0.9757,1.002,0.9506,-0.05113
t2-F1-cad-boc-2017,T2-F1,fx,0.9769,0.9734,0.9253,-0.04813
t2-F1-dkk-peg-2019,T2-F1,fx,0.7747,0.7747,0.8129,0.03818
t2-F2-whatever-it-takes-2012,T2-F2,fx,0.8385,0.8603,0.9014,0.04109


## 5 — Save

In [6]:
M19.to_parquet(DATA / "m1_9_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_9_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_9_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_9_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for name in ("m1_9_backtest_scores.parquet", "m1_9_vs_m0_per_unit.csv", "m1_9_vs_m0_per_family.csv", "m1_9_vs_m0_by_asset_type.csv"):
    print(f"data/{name:<30} {(DATA / name).stat().st_size / 1e3:8.1f} kB")

data/m1_9_backtest_scores.parquet      867.1 kB
data/m1_9_vs_m0_per_unit.csv             9.8 kB
data/m1_9_vs_m0_per_family.csv           0.5 kB
data/m1_9_vs_m0_by_asset_type.csv        0.6 kB


## 6 — In short

In [7]:
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.9 vs M1.8: better on {a['M1.9 better than M1.8']} units; per family M1.9 (M1.8): "
      + "; ".join(f"{f} {FAMILY.loc[f, 'M1.9']:.4f} ({FAMILY.loc[f, 'M1.8']:.4f})" for f in FAMILY.index if f != "all"))

  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  production walk (text off)           0.9726   (better than M0 on 74/103)
  M1.5 (EWMA sd 63)                    0.9444   (better than M0 on 85/103)
  M1.7 (recent trend)                  0.9312   (better than M0 on 86/103)
  M1.8 (+ floor, variance ratio)       0.9266   (better than M0 on 84/103)
  M1.9 (+ trend scenario)              0.9194   (better than M0 on 92/103)
M1.9 vs M1.8: better on 58/103 units; per family M1.9 (M1.8): T2-F1 0.8673 (0.8714); T2-F2 0.9461 (0.9610); T2-F3 0.9084 (0.9097); T2-F4 0.9428 (0.9498)
